# 🚀 TAMER OCR v2.1 — Unified Training Pipeline

This notebook is designed to run flawlessly on both **Google Colab** and **Kaggle**.

### 🔑 Setup Instructions:
For the best experience, set the following secrets in your environment (Colab Secrets or Kaggle Secrets):
- `HF_TOKEN`: Your Hugging Face Write Token
- `KAGGLE_KEY`: Your Kaggle API Key (Username is pre-set to `merselfares`)

*If you don't set them, the notebook will securely prompt you for them.*

In [ ]:
import os
import shutil
import gc
import sys

print("🌍 Detecting Environment...")
IS_KAGGLE = os.path.exists('/kaggle')
BASE_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
REPO_PATH = os.path.join(BASE_DIR, 'AI_PROJECT_TAMER_Complete')

os.chdir(BASE_DIR)
print(f"✅ Environment: {'Kaggle' if IS_KAGGLE else 'Google Colab'}")
print(f"📂 Base Directory: {BASE_DIR}")

print("\n🧹 Cleaning workspace for a fresh start...")
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH, ignore_errors=True)
    print(f"  - Purged old repository: {REPO_PATH}")

gc.collect()
print("✨ Workspace clean.")

In [ ]:
print("📦 Installing required dependencies...")
!pip install -q timm>=0.9.2 albumentations>=1.3.0 datasets huggingface_hub requests tqdm psutil opencv-python-headless

print("\n📥 Cloning production codebase...")
REPO_URL = "https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete"
!git clone {REPO_URL}

# CRITICAL FIX: Change working directory INTO the repo so Python finds tamer_ocr natively
os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f"✅ Environment ready. Working directory is now: {os.getcwd()}")

In [ ]:
import getpass
import os
import json

print("🔐 Configuring Authentication...")

hf_token = ""
kaggle_user = "merselfares"  # Hardcoded as requested
kaggle_key = ""

# 1. Attempt to load from environment secrets automatically
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
        kaggle_key = user_secrets.get_secret("KAGGLE_KEY")
        print("✅ Loaded credentials from Kaggle Secrets.")
    except Exception:
        pass
else:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        kaggle_key = userdata.get('KAGGLE_KEY')
        print("✅ Loaded credentials from Colab Secrets.")
    except Exception:
        pass

# 2. Fallback to manual prompts if secrets are missing
if not hf_token:
    hf_token = getpass.getpass('🔑 Enter HuggingFace Token (Write Access): ')
if not kaggle_key:
    kaggle_key = getpass.getpass(f'🔑 Enter Kaggle API Key for {kaggle_user}: ')

# 3. Apply to environment
os.environ['HF_TOKEN'] = hf_token
os.environ['KAGGLE_USERNAME'] = kaggle_user
os.environ['KAGGLE_KEY'] = kaggle_key

# 4. Write kaggle.json for the Kaggle CLI
kaggle_conf = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_conf, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_conf, 'kaggle.json')

with open(kaggle_json_path, 'w') as f:
    json.dump({"username": kaggle_user, "key": kaggle_key}, f)
os.chmod(kaggle_json_path, 0o600)

print(f"\n✅ Authentication fully configured for user: {kaggle_user}")

In [ ]:
from huggingface_hub import HfApi

# This will now work flawlessly because we are inside REPO_PATH
from tamer_ocr.config import Config
config = Config()

# Map directories to be inside the repository for clean organization
config.data_dir = os.path.join(REPO_PATH, 'data')
config.output_dir = os.path.join(REPO_PATH, 'outputs')
config.checkpoint_dir = os.path.join(REPO_PATH, 'checkpoints')
config.log_dir = os.path.join(REPO_PATH, 'logs')

for d in [config.data_dir, config.output_dir, config.checkpoint_dir, config.log_dir]:
    os.makedirs(d, exist_ok=True)

# Resolve Hugging Face Username dynamically
api = HfApi(token=os.environ['HF_TOKEN'])
hf_user = api.whoami()['name']
config.hf_token = os.environ['HF_TOKEN']
config.hf_repo_id = f"{hf_user}/tamer-ocr-model"
config.hf_dataset_repo_id = f"{hf_user}/tamer-preprocessed"

print(f"✅ Configured for HF User: {hf_user}")
print(f"✅ Model Repo: https://huggingface.co/{config.hf_repo_id}")
print(f"✅ Dataset Repo: https://huggingface.co/datasets/{config.hf_dataset_repo_id}")

In [ ]:
from tamer_ocr.core.trainer import Trainer

print("🚀 Launching TAMER Core Training Pipeline...")
print("Pipeline Order:")
print("  1. Preprocess Datasets (or pull from HF)")
print("  2. Build Swin-Transformer Model")
print("  3. Auto-Resume Checkpoints")
print("  4. Train & Evaluate")
print("="*60)

trainer = Trainer(config)

try:
    trainer.run()
except Exception as e:
    print(f"\n❌ Pipeline execution failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
from tamer_ocr.utils.checkpoint import push_checkpoint_to_hf

print("📤 Performing final artifact synchronization...")
best_pt = os.path.join(config.checkpoint_dir, 'best.pt')

if os.path.exists(best_pt):
    push_checkpoint_to_hf(best_pt, config, epoch=0, is_best=True)
    print("✅ Final best model successfully synced to HuggingFace Hub.")
else:
    print("⚠️ No best.pt artifact found. Training may have been interrupted.")